# Análise exploratória de transações financeiras

Este notebook explora a base processada pelo pipeline, compara alertas e documenta os principais sinais. Os dados são sintéticos e um alerta não representa fraude confirmada.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'transactions_scored.csv', parse_dates=['timestamp'])
df.head()

## Qualidade e visão geral

In [ ]:
summary = {
    'linhas': len(df),
    'transacoes_unicas': df['transaction_id'].nunique(),
    'contas': df['account_id'].nunique(),
    'valor_monitorado': df['amount'].sum(),
    'alertas': df['alert_flag'].sum(),
    'taxa_alertas': df['alert_flag'].mean(),
}
pd.Series(summary)

## Regras acionadas

In [ ]:
rule_columns = [column for column in df if column.startswith('rule_')]
rule_counts = df[rule_columns].sum().sort_values()
rule_counts.plot.barh(figsize=(10, 5), color='#2563EB', title='Acionamentos por regra')
plt.xlabel('Transações sinalizadas')
plt.show()

## Taxa de alertas por canal

In [ ]:
channel = df.groupby('channel').agg(
    transacoes=('transaction_id', 'count'),
    alertas=('alert_flag', 'sum'),
    valor=('amount', 'sum'),
)
channel['taxa_alertas'] = channel['alertas'] / channel['transacoes']
channel.sort_values('taxa_alertas', ascending=False)

## Avaliação contra os cenários conhecidos

In [ ]:
actual = df['is_suspicious_simulated'].eq(1)
predicted = df['alert_flag'].eq(1)
tp = (actual & predicted).sum()
fp = (~actual & predicted).sum()
fn = (actual & ~predicted).sum()
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
pd.Series({'precisao': precision, 'recall': recall, 'f1': f1})

## Conclusão

A análise deve terminar com recomendações operacionais: priorizar combinações de regras, calibrar o limiar pela capacidade da equipe, agrupar sequências em casos e registrar o desfecho das investigações.